# GeoDiff-GAN Evidence-Aware Caption Study

This notebook fine-tunes only the prompt-evidence controller from an existing trained GeoDiff-GAN checkpoint. It then compares matched, null, paraphrased, and mismatched captions with identical LR inputs and diffusion seeds. Edit outputs are not evaluated as reconstructed observations.


In [ ]:
from pathlib import Path
import os, subprocess, sys, json, shutil

REPOSITORY_URL = 'https://github.com/shashankjs2002/SI-SR-1.git'
REPOSITORY_DIR = Path('/kaggle/working/geodiff-gan-caption-study')
WORK_ROOT = Path('/kaggle/working/geodiff-caption-novelty')
DATA_ROOT = Path('/kaggle/input')
# Set explicit paths if automatic discovery selects the wrong attached dataset.
MANIFEST = None
CAPTIONS = None
PRETRAINED_CHECKPOINT = None
FAST_DEV_RUN = True
RUN_TRAINING = True
TRAIN_EPOCHS = 2 if FAST_DEV_RUN else 12
EVAL_LIMIT = 4 if FAST_DEV_RUN else 100
EVAL_STEPS = 2 if FAST_DEV_RUN else 20
SAVE_IMAGES = 4 if FAST_DEV_RUN else 12
OPTIONAL_METRICS = False  # True enables LPIPS/DISTS and may download weights
DEVICE = 'cuda'
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print('Work root:', WORK_ROOT)


## 1. Clone and install without deleting existing data


In [ ]:
def run(command, cwd=None):
    command = [str(value) for value in command]
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    if REPOSITORY_DIR.exists():
        source = REPOSITORY_DIR / 'src'
        env['PYTHONPATH'] = os.pathsep.join(filter(None, [str(source), env.get('PYTHONPATH', '')]))
    print('+', ' '.join(command), flush=True)
    subprocess.run(command, cwd=cwd, check=True, env=env)

if REPOSITORY_DIR.exists() and not (REPOSITORY_DIR / '.git').exists():
    raise RuntimeError(f'{REPOSITORY_DIR} exists but is not a Git clone; choose another REPOSITORY_DIR. Nothing was deleted.')
if REPOSITORY_DIR.exists():
    run(['git', 'pull', '--ff-only'], cwd=REPOSITORY_DIR)
else:
    run(['git', 'clone', '--depth', '1', REPOSITORY_URL, REPOSITORY_DIR])
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle.txt'], cwd=REPOSITORY_DIR)
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], cwd=REPOSITORY_DIR)
sys.path.insert(0, str(REPOSITORY_DIR / 'src'))
os.chdir(REPOSITORY_DIR)
import torch, geodiff_gan
print('Package:', geodiff_gan.__file__)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
run(['git', 'log', '-1', '--oneline'], cwd=REPOSITORY_DIR)


## 2. Discover and audit the preprocessed dataset


In [ ]:
from collections import Counter

def choose(candidates, label):
    candidates = sorted(candidates)
    if not candidates:
        raise FileNotFoundError(f'No {label} found below {DATA_ROOT}')
    print(label, 'candidates:')
    for value in candidates[:20]: print(' -', value)
    return candidates[0]

MANIFEST = Path(MANIFEST) if MANIFEST else choose(DATA_ROOT.rglob('manifest.jsonl'), 'manifest')
CAPTIONS = Path(CAPTIONS) if CAPTIONS else choose(DATA_ROOT.rglob('captions.jsonl'), 'captions')
if PRETRAINED_CHECKPOINT is None:
    checkpoints = sorted(DATA_ROOT.rglob('*joint_best.pt')) + sorted(DATA_ROOT.rglob('*joint_epoch_*.pt'))
    PRETRAINED_CHECKPOINT = checkpoints[-1] if checkpoints else None
if PRETRAINED_CHECKPOINT is None:
    raise FileNotFoundError('Attach a trained GeoDiff-GAN joint checkpoint and set PRETRAINED_CHECKPOINT.')
PRETRAINED_CHECKPOINT = Path(PRETRAINED_CHECKPOINT)
records = [json.loads(line) for line in MANIFEST.read_text(encoding='utf-8').splitlines() if line.strip()]
caption_rows = [json.loads(line) for line in CAPTIONS.read_text(encoding='utf-8').splitlines() if line.strip()]
print('Manifest:', MANIFEST)
print('Captions:', CAPTIONS, 'rows=', len(caption_rows))
print('Checkpoint:', PRETRAINED_CHECKPOINT)
print('Splits:', Counter(row['split'] for row in records))
print('Tiles:', len(set(row['tile_id'] for row in records)))
if not {'val', 'test'} <= set(row['split'] for row in records):
    raise RuntimeError('Validation and test splits are required for the caption study.')


## 3. Build the opt-in prompt-policy configuration


In [ ]:
import copy, yaml
from geodiff_gan.config import load_config

def merge(base, override):
    result = copy.deepcopy(base)
    for key, value in override.items():
        result[key] = merge(result.get(key, {}), value) if isinstance(value, dict) and isinstance(result.get(key), dict) else copy.deepcopy(value)
    return result

config = load_config(REPOSITORY_DIR / 'configs/small_12tile_improved.yaml', REPOSITORY_DIR / 'configs/default.yaml')
config = merge(config, load_config(REPOSITORY_DIR / 'configs/caption_novelty.yaml'))
config['data']['manifest'] = str(MANIFEST)
config['data']['captions'] = str(CAPTIONS)
config['training']['epochs'] = TRAIN_EPOCHS
config['training']['output_dir'] = str(WORK_ROOT / 'runs/prompt_policy')
config['training']['init_checkpoint'] = str(PRETRAINED_CHECKPOINT)
config['training']['auto_resume'] = True
config['training']['progress_mode'] = 'tqdm'
config['training']['validation_limit'] = 4 if FAST_DEV_RUN else 64
config['training']['keep_best_and_latest'] = True
CONFIG_PATH = WORK_ROOT / 'caption_novelty.yaml'
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print(CONFIG_PATH.read_text(encoding='utf-8'))


## 4. Inspect grounded captions before training


In [ ]:
from geodiff_gan.data import SentinelPatchDataset
from geodiff_gan.text import controlled_prompt_variants
from IPython.display import display
import matplotlib.pyplot as plt

preview = SentinelPatchDataset(MANIFEST, split='train', caption_file=CAPTIONS, caption_sampling='fixed', augment=False, random_degradation=False)
for index in range(min(5, len(preview))):
    sample = preview[index]
    variants = controlled_prompt_variants(sample['caption'])
    plt.figure(figsize=(5, 5)); plt.imshow(sample['hr'][:3].permute(1,2,0).clamp(0,1)); plt.axis('off'); plt.title(f'index {index}')
    plt.show()
    print(json.dumps(variants, indent=2))


## 5. Fine-tune only the prompt-evidence controller


In [ ]:
if RUN_TRAINING:
    run([sys.executable, '-m', 'geodiff_gan.cli.train', '--config', CONFIG_PATH], cwd=REPOSITORY_DIR)
RUN_DIR = Path(config['training']['output_dir'])
best = RUN_DIR / 'prompt_policy_best.pt'
latest_candidates = sorted(RUN_DIR.glob('prompt_policy_epoch_*.pt'))
EVAL_CHECKPOINT = best if best.exists() else (latest_candidates[-1] if latest_candidates else PRETRAINED_CHECKPOINT)
print('Evaluation checkpoint:', EVAL_CHECKPOINT)


## 6. Run identical-seed four-prompt ablations


In [ ]:
RESULTS = {}
for split in ('val', 'test'):
    destination = WORK_ROOT / 'evaluation' / split
    command = [sys.executable, '-m', 'geodiff_gan.cli.prompt_ablation', '--config', CONFIG_PATH, '--checkpoint', EVAL_CHECKPOINT, '--output', destination, '--split', split, '--limit', EVAL_LIMIT, '--steps', EVAL_STEPS, '--seed', 42, '--save-images', SAVE_IMAGES, '--device', DEVICE]
    if OPTIONAL_METRICS:
        command.append('--optional-metrics')
    run(command, cwd=REPOSITORY_DIR)
    RESULTS[split] = json.loads((destination / 'metrics.json').read_text(encoding='utf-8'))
print(json.dumps(RESULTS, indent=2))


## 7. Compare metrics and inspect outputs


In [ ]:
import pandas as pd
from PIL import Image
rows = []
for split, result in RESULTS.items():
    for variant, metrics in result['variants'].items(): rows.append({'split': split, 'variant': variant, **metrics})
display(pd.DataFrame(rows).round(6))
for split in RESULTS:
    paths = sorted((WORK_ROOT / 'evaluation' / split / 'images').glob('0000_*.png'))
    if paths:
        fig, axes = plt.subplots(1, len(paths), figsize=(4*len(paths), 4))
        if len(paths) == 1: axes = [axes]
        for axis, path in zip(axes, paths): axis.imshow(Image.open(path)); axis.set_title(path.stem); axis.axis('off')
        plt.tight_layout(); plt.show()


## 8. Export all evidence


In [ ]:
archive = shutil.make_archive('/kaggle/working/geodiff-caption-novelty-results', 'zip', WORK_ROOT)
print('Download:', archive)
print('Do not claim novelty until matched utility, mismatch suppression, LR consistency, and unseen-tile tests all pass.')
